# Adaptive Heuristic Project Portfolio Selection

This notebook implements a hierarchical project portfolio selection model with:

## Decision Levels
- Level 1: Feasibility screening (Kill / Consider)
- Level 2: Primary prioritization score
- Level 3: Complementary score (secondary refinement)
- Level 4: Future capability score (secondary refinement)

## Portfolio-Level Constraints
- Budget constraint
- Staff-hours constraint
- Time horizon balance
- Initiative intent balance

## Algorithms Implemented
- Greedy baseline
- Simulated Annealing (SA)
- Adaptive Simulated Annealing (Adaptive SA)


## 1. Imports

In [1]:
import random
import math
import time
import statistics

random.seed(42)  # Reproducibility


## 2. Project Data Input

In [2]:
USE_SYNTHETIC = True   # Set False to enter projects manually
N_PROJECTS = 70        # Used only when USE_SYNTHETIC=True

class Project:
    def __init__(self, idx, g, p, c, f, b, h, time_cat, intent_cat):
        self.idx = idx
        self.g = g              # screening pass/fail
        self.p = p              # primary prioritization score
        self.c = c              # complementary score
        self.f = f              # future capability score
        self.b = b              # budget requirement
        self.h = h              # staff-hours requirement
        self.time_cat = time_cat
        self.intent_cat = intent_cat

def generate_projects(N=50):
    projects = []
    time_categories = ["Short", "Medium", "Long"]
    intent_categories = ["Exploratory", "Exponential", "Sustaining"]

    for i in range(N):
        g = 1 if random.random() < 0.85 else 0
        p = random.randint(50, 100)
        c = random.randint(0, 10)
        # mild bias: long projects tend to have higher future capability
        time_cat = random.choice(time_categories)
        if time_cat == "Long":
            f = random.randint(3, 10)
        else:
            f = random.randint(0, 8)
        b = random.randint(50, 200)
        # correlate hours with budget a bit (simple, still synthetic)
        h = max(40, int(2.0 * b + random.randint(-50, 80)))
        intent_cat = random.choice(intent_categories)

        projects.append(Project(i, g, p, c, f, b, h, time_cat, intent_cat))

    return projects

if USE_SYNTHETIC:
    projects = generate_projects(N_PROJECTS)
else:
    projects = [
        # Project(idx, g, p, c, f, b, h, time_cat, intent_cat)
        Project(0, 1, 85, 6, 8, 120, 200, "Short", "Exploratory"),
        Project(1, 1, 90, 4, 9, 150, 250, "Medium", "Exponential"),
        Project(2, 1, 70, 3, 5, 100, 180, "Long", "Sustaining"),
    ]

print("Number of candidate projects:", len(projects))
print("Eligible projects (g_i=1):", sum(1 for p in projects if p.g == 1))


Number of candidate projects: 70
Eligible projects (g_i=1): 61


## 3. Resource Constraints

In [3]:
USE_AUTO_CONSTRAINTS = True

if USE_AUTO_CONSTRAINTS:
    eligible_projects = [p for p in projects if p.g == 1]
    B_max = 0.35 * sum(p.b for p in eligible_projects)
    H_max = 0.35 * sum(p.h for p in eligible_projects)
else:
    B_max = 5000
    H_max = 8000

print("Budget limit B_max:", B_max)
print("Hours limit  H_max:", H_max)


Budget limit B_max: 2801.0499999999997
Hours limit  H_max: 5658.45


## 4. Mathematical Model Parameters

In [4]:
EPSILON = 1e-6
LAMBDA_TIME = 1.0
LAMBDA_INTENT = 1.0


## 5. Heuristic Parameters

In [5]:
MAX_ITER = 20000
NO_IMPROVEMENT_LIMIT = 2000

INITIAL_T = 100

# Standard SA cooling
ALPHA = 0.95

# Adaptive cooling
ALPHA_FAST = 0.90
ALPHA_SLOW = 0.98

WINDOW_SIZE = 200
RHO_MIN = 0.20
RHO_MAX = 0.40
REPAIR_THRESHOLD = 0.30


## 6. Objective and Penalty Functions

In [6]:
def compute_time_penalty(solution, projects, target):
    counts = {"Short": 0, "Medium": 0, "Long": 0}
    for x, p in zip(solution, projects):
        if x == 1:
            counts[p.time_cat] += 1
    return sum(abs(counts[k] - target[k]) for k in counts)

def compute_intent_penalty(solution, projects, target):
    counts = {"Exploratory": 0, "Exponential": 0, "Sustaining": 0}
    for x, p in zip(solution, projects):
        if x == 1:
            counts[p.intent_cat] += 1
    return sum(abs(counts[k] - target[k]) for k in counts)

def objective(solution, projects, target_time, target_intent):
    primary = sum(p.p * x for p, x in zip(projects, solution))
    secondary = sum((p.c + p.f) * x for p, x in zip(projects, solution))
    time_pen = compute_time_penalty(solution, projects, target_time)
    intent_pen = compute_intent_penalty(solution, projects, target_intent)

    return primary + EPSILON * secondary - LAMBDA_TIME * time_pen - LAMBDA_INTENT * intent_pen


## 7. Feasibility and Repair

In [7]:
def is_feasible(solution, projects, B_max, H_max):
    total_b = sum(p.b * x for p, x in zip(projects, solution))
    total_h = sum(p.h * x for p, x in zip(projects, solution))
    return total_b <= B_max and total_h <= H_max

def repair(solution, projects, B_max, H_max):
    # Simple repair: remove randomly selected projects until feasible
    while not is_feasible(solution, projects, B_max, H_max):
        selected = [i for i, x in enumerate(solution) if x == 1]
        if not selected:
            break
        solution[random.choice(selected)] = 0
    return solution


## 8. Greedy Baseline and Dynamic Targets

In [8]:
def greedy(projects, B_max, H_max):
    solution = [0] * len(projects)
    sorted_indices = sorted(range(len(projects)), key=lambda i: projects[i].p, reverse=True)

    for i in sorted_indices:
        if projects[i].g == 0:
            continue
        solution[i] = 1
        if not is_feasible(solution, projects, B_max, H_max):
            solution[i] = 0

    return solution

def compute_targets(solution):
    K = sum(solution)
    target_time = {"Short": round(0.3 * K), "Medium": round(0.4 * K), "Long": round(0.3 * K)}
    target_intent = {"Exploratory": round(0.3 * K), "Exponential": round(0.4 * K), "Sustaining": round(0.3 * K)}
    return target_time, target_intent

greedy_sol = greedy(projects, B_max, H_max)
target_time, target_intent = compute_targets(greedy_sol)

print("Greedy selects K =", sum(greedy_sol), "projects")
print("Target time mix:", target_time)
print("Target intent mix:", target_intent)


Greedy selects K = 20 projects
Target time mix: {'Short': 6, 'Medium': 8, 'Long': 6}
Target intent mix: {'Exploratory': 6, 'Exponential': 8, 'Sustaining': 6}


## 9. Neighborhood Operator

In [9]:
def generate_neighbor(solution, projects, swap_prob=0.6):
    new_solution = solution[:]

    if random.random() < swap_prob:
        # Swap move
        selected = [i for i, x in enumerate(solution) if x == 1]
        unselected = [i for i, x in enumerate(solution) if x == 0 and projects[i].g == 1]
        if selected and unselected:
            i = random.choice(selected)
            j = random.choice(unselected)
            new_solution[i] = 0
            new_solution[j] = 1
    else:
        # Flip move
        eligible = [i for i, p in enumerate(projects) if p.g == 1]
        i = random.choice(eligible)
        new_solution[i] = 1 - new_solution[i]

    return new_solution


## 10. Standard Simulated Annealing (SA)

In [10]:
def simulated_annealing(projects, B_max, H_max, target_time, target_intent):
    current = greedy(projects, B_max, H_max)
    current_value = objective(current, projects, target_time, target_intent)

    best = current[:]
    best_value = current_value

    T = INITIAL_T
    no_improvement = 0
    start_time = time.time()

    for _ in range(MAX_ITER):
        candidate = generate_neighbor(current, projects)
        candidate = repair(candidate, projects, B_max, H_max)

        candidate_value = objective(candidate, projects, target_time, target_intent)
        delta = candidate_value - current_value

        if delta >= 0 or random.random() < math.exp(delta / T):
            current = candidate
            current_value = candidate_value

            if candidate_value > best_value:
                best = candidate[:]
                best_value = candidate_value
                no_improvement = 0
            else:
                no_improvement += 1
        else:
            no_improvement += 1

        T *= ALPHA

        if no_improvement >= NO_IMPROVEMENT_LIMIT:
            break

    runtime = time.time() - start_time
    return best, best_value, runtime


## 11. Adaptive Simulated Annealing (ASA)

In [11]:
def adaptive_simulated_annealing(projects, B_max, H_max, target_time, target_intent):
    current = greedy(projects, B_max, H_max)
    current_value = objective(current, projects, target_time, target_intent)

    best = current[:]
    best_value = current_value

    T = INITIAL_T
    no_improvement = 0

    accept_count = 0
    repair_count = 0
    swap_prob = 0.6

    start_time = time.time()

    for iteration in range(MAX_ITER):
        candidate = generate_neighbor(current, projects, swap_prob)

        if not is_feasible(candidate, projects, B_max, H_max):
            repair_count += 1
            candidate = repair(candidate, projects, B_max, H_max)

        candidate_value = objective(candidate, projects, target_time, target_intent)
        delta = candidate_value - current_value

        if delta >= 0 or random.random() < math.exp(delta / T):
            current = candidate
            current_value = candidate_value
            accept_count += 1

            if candidate_value > best_value:
                best = candidate[:]
                best_value = candidate_value
                no_improvement = 0
            else:
                no_improvement += 1
        else:
            no_improvement += 1

        # Adaptive updates every WINDOW_SIZE iterations
        if iteration > 0 and iteration % WINDOW_SIZE == 0:
            acceptance_rate = accept_count / WINDOW_SIZE
            repair_rate = repair_count / WINDOW_SIZE

            # Adaptive cooling
            if acceptance_rate > RHO_MAX:
                T *= ALPHA_FAST
            elif acceptance_rate < RHO_MIN:
                T *= ALPHA_SLOW
            else:
                T *= ALPHA

            # Adaptive move mixing
            swap_prob = 0.8 if repair_rate > REPAIR_THRESHOLD else 0.6

            accept_count = 0
            repair_count = 0
        else:
            T *= ALPHA

        if no_improvement >= NO_IMPROVEMENT_LIMIT:
            break

    runtime = time.time() - start_time
    return best, best_value, runtime


## 12. Comparison: Greedy vs SA vs ASA

In [12]:
def run_comparison(projects, B_max, H_max, target_time, target_intent, runs=10):
    greedy_sol = greedy(projects, B_max, H_max)
    greedy_val = objective(greedy_sol, projects, target_time, target_intent)

    print("Greedy objective:", greedy_val)
    print("--------------------------------------------------")

    sa_values, sa_times = [], []
    for _ in range(runs):
        _, val, runtime = simulated_annealing(projects, B_max, H_max, target_time, target_intent)
        sa_values.append(val)
        sa_times.append(runtime)

    ada_values, ada_times = [], []
    for _ in range(runs):
        _, val, runtime = adaptive_simulated_annealing(projects, B_max, H_max, target_time, target_intent)
        ada_values.append(val)
        ada_times.append(runtime)

    print("Standard SA:")
    print("  Avg Objective:", sum(sa_values)/runs)
    print("  Std Dev:", statistics.stdev(sa_values) if runs > 1 else 0.0)
    print("  Avg Runtime:", sum(sa_times)/runs)

    print("--------------------------------------------------")

    print("Adaptive SA:")
    print("  Avg Objective:", sum(ada_values)/runs)
    print("  Std Dev:", statistics.stdev(ada_values) if runs > 1 else 0.0)
    print("  Avg Runtime:", sum(ada_times)/runs)

    print("--------------------------------------------------")

    if sum(ada_values)/runs >= sum(sa_values)/runs:
        print("Adaptive SA outperforms or equals Standard SA ✔")
    else:
        print("Adaptive SA underperforms Standard SA ❌")

run_comparison(projects, B_max, H_max, target_time, target_intent, runs=10)


Greedy objective: 1824.000196
--------------------------------------------------
Standard SA:
  Avg Objective: 2101.4002213
  Std Dev: 48.63287643102876
  Avg Runtime: 0.7203778743743896
--------------------------------------------------
Adaptive SA:
  Avg Objective: 2098.5002194
  Std Dev: 67.86466338726491
  Avg Runtime: 0.3054773807525635
--------------------------------------------------
Adaptive SA underperforms Standard SA ❌


In [13]:
run_comparison(projects, B_max, H_max, target_time, target_intent)


Greedy objective: 1824.000196
--------------------------------------------------
Standard SA:
  Avg Objective: 2111.7002215
  Std Dev: 54.25055891058858
  Avg Runtime: 0.20904779434204102
--------------------------------------------------
Adaptive SA:
  Avg Objective: 2120.0002234
  Std Dev: 35.54027954307681
  Avg Runtime: 0.296049976348877
--------------------------------------------------
Adaptive SA outperforms or equals Standard SA ✔


## 13. Final Portfolio Output (Adaptive SA)

In [14]:
def extract_selected_projects(solution, projects):
    return [projects[i] for i, x in enumerate(solution) if x == 1]

ada_sol, ada_val, ada_runtime = adaptive_simulated_annealing(projects, B_max, H_max, target_time, target_intent)
selected_projects = sorted(extract_selected_projects(ada_sol, projects), key=lambda p: p.p, reverse=True)

print("Adaptive SA best objective (single run):", ada_val)
print("Adaptive SA runtime (single run):", ada_runtime)
print("Feasible:", is_feasible(ada_sol, projects, B_max, H_max))
print("Number of selected projects:", len(selected_projects))

total_budget = sum(p.b for p in selected_projects)
total_hours = sum(p.h for p in selected_projects)
print("Total budget used:", total_budget, "| Limit:", B_max)
print("Total hours used:", total_hours, "| Limit:", H_max)

print("\nSelected Projects:")
for p in selected_projects:
    print(f"Project {p.idx:>2} | g={p.g} | p={p.p:>3} | c={p.c:>2} | f={p.f:>2} | b={p.b:>3} | h={p.h:>3} | Time={p.time_cat:<6} | Intent={p.intent_cat}")


Adaptive SA best objective (single run): 2103.000222
Adaptive SA runtime (single run): 0.3903970718383789
Feasible: True
Number of selected projects: 24
Total budget used: 2789 | Limit: 2801.0499999999997
Total hours used: 5623 | Limit: 5658.45

Selected Projects:
Project 15 | g=1 | p=100 | c=10 | f= 1 | b=148 | h=343 | Time=Medium | Intent=Sustaining
Project  4 | g=1 | p= 98 | c= 5 | f= 1 | b=147 | h=268 | Time=Short  | Intent=Exponential
Project 40 | g=1 | p= 98 | c= 1 | f= 7 | b=194 | h=363 | Time=Short  | Intent=Exploratory
Project 34 | g=1 | p= 96 | c= 8 | f=10 | b= 89 | h=176 | Time=Long   | Intent=Exponential
Project 52 | g=1 | p= 94 | c= 1 | f= 0 | b=170 | h=346 | Time=Medium | Intent=Exploratory
Project 67 | g=1 | p= 94 | c= 0 | f= 6 | b=106 | h=207 | Time=Short  | Intent=Sustaining
Project  9 | g=1 | p= 93 | c=10 | f= 2 | b=186 | h=384 | Time=Short  | Intent=Exploratory
Project 63 | g=1 | p= 93 | c= 3 | f=10 | b= 93 | h=157 | Time=Long   | Intent=Exponential
Project  7 | g=1 

In [15]:
print("\nFinal Portfolio (Adaptive SA):")

selected_projects = extract_selected_projects(ada_sol, projects)

print("Number of selected projects:", len(selected_projects))

for p in selected_projects:
    print(f"Project {p.idx} | Priority={p.p} | Budget={p.b} | Hours={p.h} | Time={p.time_cat} | Intent={p.intent_cat}")



Final Portfolio (Adaptive SA):
Number of selected projects: 24
Project 1 | Priority=84 | Budget=58 | Hours=73 | Time=Long | Intent=Exploratory
Project 4 | Priority=98 | Budget=147 | Hours=268 | Time=Short | Intent=Exponential
Project 7 | Priority=92 | Budget=109 | Hours=193 | Time=Medium | Intent=Exponential
Project 8 | Priority=90 | Budget=140 | Hours=283 | Time=Short | Intent=Sustaining
Project 9 | Priority=93 | Budget=186 | Hours=384 | Time=Short | Intent=Exploratory
Project 15 | Priority=100 | Budget=148 | Hours=343 | Time=Medium | Intent=Sustaining
Project 17 | Priority=68 | Budget=50 | Hours=117 | Time=Short | Intent=Sustaining
Project 22 | Priority=88 | Budget=101 | Hours=231 | Time=Short | Intent=Exponential
Project 25 | Priority=90 | Budget=58 | Hours=150 | Time=Short | Intent=Exploratory
Project 28 | Priority=92 | Budget=155 | Hours=379 | Time=Medium | Intent=Sustaining
Project 29 | Priority=91 | Budget=153 | Hours=342 | Time=Short | Intent=Exploratory
Project 34 | Priority=

In [16]:
print("\nFinal Portfolio (Adaptive SA):")

selected_projects = sorted(
    extract_selected_projects(ada_sol, projects),
    key=lambda p: p.p,
    reverse=True
)


print("Number of selected projects:", len(selected_projects))

for p in selected_projects:
    print(f"Project {p.idx} | Priority={p.p} | Budget={p.b} | Hours={p.h} | Time={p.time_cat} | Intent={p.intent_cat}")



Final Portfolio (Adaptive SA):
Number of selected projects: 24
Project 15 | Priority=100 | Budget=148 | Hours=343 | Time=Medium | Intent=Sustaining
Project 4 | Priority=98 | Budget=147 | Hours=268 | Time=Short | Intent=Exponential
Project 40 | Priority=98 | Budget=194 | Hours=363 | Time=Short | Intent=Exploratory
Project 34 | Priority=96 | Budget=89 | Hours=176 | Time=Long | Intent=Exponential
Project 52 | Priority=94 | Budget=170 | Hours=346 | Time=Medium | Intent=Exploratory
Project 67 | Priority=94 | Budget=106 | Hours=207 | Time=Short | Intent=Sustaining
Project 9 | Priority=93 | Budget=186 | Hours=384 | Time=Short | Intent=Exploratory
Project 63 | Priority=93 | Budget=93 | Hours=157 | Time=Long | Intent=Exponential
Project 7 | Priority=92 | Budget=109 | Hours=193 | Time=Medium | Intent=Exponential
Project 28 | Priority=92 | Budget=155 | Hours=379 | Time=Medium | Intent=Sustaining
Project 29 | Priority=91 | Budget=153 | Hours=342 | Time=Short | Intent=Exploratory
Project 8 | Prior